# Credit Card Fraud Detection - Transformasi Data dan Feature Selection

Notebook ini dibuat untuk memenuhi bagian tugas berikut:

1. **Transformasi Data**
   - Jika ada data kategorikal, lakukan encoding menggunakan Label Encoding atau One-Hot Encoding.
   - Jika ada variabel numerik dengan skala berbeda, lakukan normalisasi atau standardisasi.

2. **Feature Selection & Extraction**
   - Pilih fitur yang paling relevan menggunakan PCA atau Feature Importance dari Decision Tree/Random Forest.
   - Jika dataset memiliki terlalu banyak fitur redundan, lakukan reduksi dimensi.
   - Jika perlu, buat fitur baru berdasarkan transformasi data yang ada.


## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)

## 2. Load Dataset

In [ ]:
df = pd.read_csv('creditcard.csv')

print('Shape dataset awal:', df.shape)
display(df.head())
display(df.info())

## 3. Basic Cleaning Sesuai Tugas Sebelumnya

Tahap ini mengikuti preprocessing dasar yang sudah dilakukan sebelumnya, yaitu mengecek missing values, menghapus duplikasi, dan menangani outlier pada kolom `Amount` menggunakan metode IQR.

In [ ]:
print('Total missing values:', df.isnull().sum().sum())
print('Jumlah duplikasi:', df.duplicated().sum())

df = df.drop_duplicates().copy()
print('Shape setelah hapus duplikasi:', df.shape)

In [ ]:
# Handling outlier pada Amount dengan metode IQR
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

before_shape = df.shape
df = df[(df['Amount'] >= lower_bound) & (df['Amount'] <= upper_bound)].copy()

print('Shape sebelum hapus outlier Amount:', before_shape)
print('Shape setelah hapus outlier Amount:', df.shape)
print('Lower bound:', lower_bound)
print('Upper bound:', upper_bound)

## 4. Transformasi Data: Encoding Data Kategorikal

Requirement meminta encoding jika ada data kategorikal. Oleh karena itu, pertama kita cek tipe data setiap kolom.

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print('Kolom kategorikal:', categorical_cols)
print('Jumlah kolom kategorikal:', len(categorical_cols))
print('Jumlah kolom numerik:', len(numeric_cols))

In [ ]:
# Encoding hanya dilakukan jika terdapat kolom kategorikal.
# Pada dataset Credit Card Fraud Detection, seluruh fitur sudah numerik.

df_encoded = df.copy()

if len(categorical_cols) > 0:
    # Contoh pendekatan One-Hot Encoding untuk kolom kategorikal nominal
    df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=True)
    print('Encoding dilakukan menggunakan One-Hot Encoding.')
else:
    print('Tidak ada kolom kategorikal, sehingga Label Encoding / One-Hot Encoding tidak diperlukan.')

print('Shape setelah proses encoding:', df_encoded.shape)

### Interpretasi Encoding

Dataset ini tidak memiliki fitur kategorikal. Semua kolom sudah berbentuk numerik, yaitu `Time`, `Amount`, `V1` sampai `V28`, dan target `Class`. Karena itu, encoding tidak perlu dilakukan. Kolom `Class` juga sudah berbentuk label numerik, yaitu `0` untuk transaksi normal dan `1` untuk transaksi fraud.

## 5. Standardisasi Variabel Numerik

Kolom `Time` dan `Amount` memiliki skala yang berbeda dibandingkan fitur `V1` sampai `V28`. Karena itu, `Time` dan `Amount` distandardisasi menggunakan `StandardScaler`.

In [ ]:
scaler = StandardScaler()

df_transformed = df_encoded.copy()
df_transformed['Amount_scaled'] = scaler.fit_transform(df_transformed[['Amount']])
df_transformed['Time_scaled'] = scaler.fit_transform(df_transformed[['Time']])

# Drop kolom asli karena sudah diganti dengan versi scaled
df_transformed = df_transformed.drop(['Amount', 'Time'], axis=1)

print('Shape setelah standardisasi:', df_transformed.shape)
display(df_transformed.head())

In [ ]:
print('Mean Amount_scaled:', round(df_transformed['Amount_scaled'].mean(), 4))
print('Std Amount_scaled:', round(df_transformed['Amount_scaled'].std(), 4))
print('Mean Time_scaled:', round(df_transformed['Time_scaled'].mean(), 4))
print('Std Time_scaled:', round(df_transformed['Time_scaled'].std(), 4))

### Interpretasi Standardisasi

Standardisasi membuat `Amount_scaled` dan `Time_scaled` memiliki rata-rata mendekati 0 dan standar deviasi mendekati 1. Hal ini penting agar fitur dengan skala besar tidak mendominasi proses analisis, khususnya pada metode seperti PCA.

## 6. Feature Engineering

Karena fitur `V1` sampai `V28` merupakan fitur anonim hasil PCA dari dataset asli, feature engineering yang terlalu kompleks sulit diinterpretasikan. Oleh karena itu, dibuat fitur sederhana dari `Amount_scaled`, yaitu `Amount_abs`, untuk menangkap transaksi yang nilai amount-nya jauh dari rata-rata.

In [ ]:
df_transformed['Amount_abs'] = df_transformed['Amount_scaled'].abs()

print('Shape setelah feature engineering:', df_transformed.shape)
display(df_transformed[['Amount_scaled', 'Amount_abs', 'Class']].head())

## 7. Feature Extraction dan Reduksi Dimensi dengan PCA

PCA digunakan untuk melihat apakah fitur-fitur dapat direduksi ke jumlah komponen yang lebih kecil tanpa kehilangan terlalu banyak informasi. Walaupun `V1` sampai `V28` sudah merupakan hasil PCA dari data asli, PCA tambahan tetap dapat digunakan untuk analisis reduksi dimensi pada dataset yang sudah diproses.

In [ ]:
X = df_transformed.drop('Class', axis=1)
y = df_transformed['Class']

# Standardisasi seluruh fitur input sebelum PCA
pca_scaler = StandardScaler()
X_scaled = pca_scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

pca_summary = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(len(explained_variance))],
    'Explained Variance Ratio': explained_variance,
    'Cumulative Variance': cumulative_variance
})

display(pca_summary.head(15))

In [ ]:
n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print('Jumlah komponen untuk menjelaskan >= 90% variance:', n_components_90)
print('Jumlah komponen untuk menjelaskan >= 95% variance:', n_components_95)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o')
plt.axhline(y=0.90, color='red', linestyle='--', label='90% variance')
plt.axhline(y=0.95, color='green', linestyle='--', label='95% variance')
plt.xlabel('Jumlah Komponen PCA')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance PCA')
plt.legend()
plt.show()

In [ ]:
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    'PC1': X_pca_2d[:, 0],
    'PC2': X_pca_2d[:, 1],
    'Class': y.values
})

pca_sample = pca_df.sample(n=min(10000, len(pca_df)), random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=pca_sample, x='PC1', y='PC2', hue='Class', alpha=0.6, palette={0: 'blue', 1: 'red'})
plt.title('Visualisasi PCA 2D: PC1 vs PC2')
plt.show()

### Interpretasi PCA

PCA membantu melihat seberapa banyak informasi yang dapat dijelaskan oleh sejumlah komponen utama. Jika 90% atau 95% variance dapat dijelaskan oleh jauh lebih sedikit komponen dibandingkan jumlah fitur awal, maka reduksi dimensi dapat dipertimbangkan. Scatter plot PC1 dan PC2 digunakan untuk melihat apakah transaksi fraud dan normal memiliki kecenderungan terpisah dalam ruang dua dimensi.

## 8. Feature Selection dengan Random Forest Feature Importance

Random Forest digunakan untuk menghitung feature importance, yaitu seberapa besar kontribusi setiap fitur dalam memisahkan transaksi normal dan fraud. Karena dataset sangat imbalance, training feature importance dilakukan pada sampel yang diseimbangkan agar kelas fraud tetap terwakili.

In [ ]:
# Membuat balanced sample untuk mempercepat proses dan mengurangi bias terhadap kelas mayoritas
rf_data = df_transformed.copy()
fraud_df = rf_data[rf_data['Class'] == 1]
normal_df = rf_data[rf_data['Class'] == 0]

if len(fraud_df) > 0:
    normal_sample_size = min(len(normal_df), len(fraud_df) * 5)
    normal_sample = normal_df.sample(n=normal_sample_size, random_state=42)
    rf_sample = pd.concat([fraud_df, normal_sample]).sample(frac=1, random_state=42)
else:
    rf_sample = rf_data.sample(n=min(10000, len(rf_data)), random_state=42)

print('Distribusi Class pada sample Random Forest:')
print(rf_sample['Class'].value_counts())

X_rf = rf_sample.drop('Class', axis=1)
y_rf = rf_sample['Class']

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_model.fit(X_rf, y_rf)

feature_importance = pd.DataFrame({
    'Feature': X_rf.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

display(feature_importance.head(15))

In [ ]:
top_n = 15
top_features = feature_importance.head(top_n)

plt.figure(figsize=(10, 7))
sns.barplot(data=top_features, x='Importance', y='Feature', palette='viridis')
plt.title(f'Top {top_n} Feature Importance - Random Forest')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## 9. Pemilihan Fitur Akhir

Fitur akhir dipilih berdasarkan ranking feature importance dari Random Forest. Contoh berikut memilih 10 fitur teratas sebagai fitur yang paling relevan untuk tahap analisis atau modeling berikutnya.

In [ ]:
selected_features = feature_importance.head(10)['Feature'].tolist()
df_selected = df_transformed[selected_features + ['Class']].copy()

print('Fitur terpilih:')
print(selected_features)
print('
Shape dataset dengan fitur terpilih:', df_selected.shape)
display(df_selected.head())

## 10. Kesimpulan dan Interpretasi

1. **Encoding Data Kategorikal**
   - Dataset tidak memiliki kolom kategorikal.
   - Karena semua fitur sudah numerik, Label Encoding atau One-Hot Encoding tidak diperlukan.

2. **Standardisasi Variabel Numerik**
   - `Amount` dan `Time` memiliki skala berbeda dibandingkan fitur lainnya.
   - Keduanya distandardisasi menjadi `Amount_scaled` dan `Time_scaled`.
   - Standardisasi membuat fitur lebih seimbang untuk analisis seperti PCA.

3. **Feature Engineering**
   - Fitur baru `Amount_abs` dibuat dari nilai absolut `Amount_scaled`.
   - Fitur ini membantu menangkap transaksi dengan nilai amount yang jauh dari rata-rata.
   - Feature engineering dibatasi karena `V1` sampai `V28` adalah fitur anonim hasil PCA.

4. **PCA / Reduksi Dimensi**
   - PCA digunakan untuk melihat berapa banyak komponen yang dibutuhkan untuk menjelaskan mayoritas variance dataset.
   - Jika jumlah komponen untuk mencapai 90% atau 95% variance jauh lebih sedikit dari jumlah fitur awal, reduksi dimensi dapat dipertimbangkan.
   - Visualisasi PC1 dan PC2 membantu melihat pola pemisahan awal antara transaksi normal dan fraud.

5. **Feature Importance Random Forest**
   - Random Forest digunakan untuk memilih fitur yang paling relevan terhadap target `Class`.
   - Fitur dengan importance tertinggi dapat digunakan sebagai kandidat fitur utama untuk tahap modeling.
   - Karena dataset sangat imbalance, hasil feature importance perlu divalidasi lagi pada tahap modeling dan evaluasi.
